In [ ]:
# Load the package from the repository root, including when this notebook
# is opened from HODDIES/ or HODDIES/sim_loader/.
from pathlib import Path
import os
import sys
import tempfile

repo_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / "HODDIES" / "__init__.py").is_file()), None)
if repo_root is not None and str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
# Keep Numba caches consistent with the package-qualified module names.
os.environ.setdefault("NUMBA_CACHE_DIR", str(Path(tempfile.gettempdir()) / "hoddies-tutorial-numba"))

from HODDIES.sim_loader import AbacusSummitSim


In [ ]:

default_abacus_param = {'boxsize': None,  # Simulation box size (to be set if path_to_sim is provided)
    'path_to_sim': '/dvs_ro/cfs/cdirs/desi/cosmosim/Abacus',  # Path to halo catalogs
    'mass_cut': None,  # Minimum halo mass cut in log10(Msun/h) e.g. 12 to cut M < 10**12
    'z_simu': 0.5,  # Redshift of simulation snapshot (list of redshifts from Abacus light cone shells.)
                      # Redshift available for Uchuu snapshot [0.43, 0.49, 0.56, 0.63, 0.70, 0.78, 0.86, 0.94, 1.03, 1.12, 1.22, 1.32, 1.43, 1.65]
    'sim_name': 'AbacusSummit_highbase_c000_ph100',  # Specific to Abacus simulation
    'load_particles': True,  # Whether to load particle/subhalos data
}
test=AbacusSummitSim(**default_abacus_param)


[001315.09]  08-11 07:48 AbacusSummitSim              INFO     Initializing AbacusSummitSim.
[001315.09]  08-11 07:48 AbacusSummitSim              INFO     Load Compaso catalogue from /dvs_ro/cfs/cdirs/desi/users/arocher/HODDIES_data/Uchuu/AbacusSummit_highbase_c000_ph100/halos/z0.500 with particles...


FileNotFoundError: Path "/dvs_ro/cfs/cdirs/desi/users/arocher/HODDIES_data/Uchuu/AbacusSummit_highbase_c000_ph100/halos/z0.500" does not exist!

In [ ]:
from HODDIES.sim_loader.Uchuu_io import UchuuSim

default_Uchuu_param = {'path_to_sim': '/dvs_ro/cfs/cdirs/desi/users/arocher/HODDIES_data/Uchuu/',  # Path to halo catalogs
    'mass_cut': 11.5,  # Minimum halo mass cut in log10(Msun/h) e.g. 12 to cut M < 10**12
    'z_simu': 1.65,  # Redshift of simulation snapshot
    'sim_name': 'Uchuu2Gpc',  # Specific to Uchuu simulation
    'load_particles': True,  # Whether to load particle/subhalos data
}
test=UchuuSim(**default_Uchuu_param, nthreads=256)

In [ ]:
test.hcat

Catalog(csize=73221666, size=73221666, columns=['Mh', 'Rh', 'Rs', 'Vrms', 'c', 'halo_id', 'log10_Mh', 'vx', 'vy', 'vz', 'x', 'y', 'z'])

In [ ]:
# Test with custum pre loaded catalogue
import numpy as np

np.random.normal(5, 2, 100)

array([ 5.1487346 ,  7.46576548,  4.73976164,  4.90795676,  5.69986631,
        6.3939382 ,  4.5356108 ,  6.44951793,  5.81819496,  8.15256018,
        3.9293019 ,  7.44788637,  5.63062818,  4.7610759 ,  3.48102591,
        5.37564637,  2.74596147,  5.90687008,  4.34118571,  3.5089553 ,
        4.63632263,  7.99129375,  1.53015646,  2.16010117,  9.88598165,
        5.09303173,  5.62481747,  4.55606257,  5.24674514,  7.44106502,
        4.02674554,  6.61989066,  7.78420929,  3.16126282,  2.32127705,
        3.12410794,  4.73255442, -1.64315173,  4.15935297,  2.78361249,
        2.86921102,  5.12240607,  1.97827526,  5.66804879,  7.36514033,
        2.98304811,  2.75117396,  5.71978975,  5.02324888,  7.20618688,
        6.11648834,  4.63993455,  3.4043022 ,  6.61315397,  5.352319  ,
        6.74221601,  5.05516458,  2.42481284,  5.88215327,  2.04679258,
        9.00301303,  2.09914151,  3.18745618,  3.01948552,  6.37249938,
        5.61407351,  4.20063154,  4.72725805,  3.62924352,  6.21

In [1]:
import numpy as np

def generate_rd_cat(N, boxsize=1000, seed=42, particle_data=False):
    """
    Generate a random catalogue of N points in a unit cube.

    Parameters
    ----------
    N : int
        Number of points to generate.
    boxsize : float, optional
        Size of the box in which to generate points. Default is 1000 Mpc/h.
    seed : int, optional
        Random seed for reproducibility. Default is 42.

    Returns
    -------
    data : dict
        Dictionary containing the generated data with the columns required for the HODDIES code.
       
    """
    np.random.seed(seed)
    data = {}
    
    data['vx'] = np.random.normal(0, 300, N)
    data['vy'] = np.random.normal(0, 300, N)
    data['vz'] = np.random.normal(0, 300, N)
    data['x'] = np.random.uniform(0, boxsize, N)
    data['y'] = np.random.uniform(0, boxsize, N)
    data['z'] = np.random.uniform(0, boxsize, N)
    data['halo_id'] = np.arange(N)
    if particle_data:
        return data
    data['Mh'] = 10**np.random.uniform(11, 15, N)
    data['Rh'] = np.random.uniform(0.1, 1.0, N)
    data['Rs'] = np.random.uniform(0.01, 0.1, N)
    data['Vrms'] = np.random.uniform(100, 1000, N)
    data['c'] = np.random.normal(5, 2, N)
    data['c'][data['c'] < 0] = 5
    data['log10_Mh'] = np.log10(data['Mh'])
    return data
    

In [ ]:
from HODDIES.sim_loader import Base_catalogue
boxsize = 1000
cat_dict = generate_rd_cat(1_000_000, boxsize=boxsize, particle_data=False, seed=1)
part_dict = generate_rd_cat(1_000_000, particle_data=True, seed=2, boxsize=boxsize)

tt = Base_catalogue(halo_data=cat_dict, particle_data=part_dict, boxsize=1000)

In [6]:
tt.hcat['x']

array([121.85228851 874.85194342   2.29522938 ... 416.30264486  28.25105448
 788.85217707])